In [3]:
pip install opencv-python

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install npy-append-array

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install -U ultralytics

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Requirement already up-to-date: ultralytics in c:\users\polis\anaconda3\lib\site-packages (8.4.27)
Note: you may need to restart the kernel to use updated packages.


In [27]:
pip install -U ultralytics numpy

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Requirement already up-to-date: numpy in c:\users\polis\anaconda3\lib\site-packages (1.24.4)
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.27
    Uninstalling ultralytics-8.4.27:
      Successfully uninstalled ultralytics-8.4.27
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import numpy as np
import cv2
import os
from tqdm import tqdm
# from npy_append_array import NpyAppendArray
from torch.utils.data import Dataset, DataLoader
np.int = int 
np.float = float
np.bool = bool
import pandas as pd

print(torch.cuda.is_available())

True


Preprocess Labels

In [19]:
def click_event(event, x, y, flags, params):
    # Check if the left mouse button was clicked
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Clicked at: {x}, {y}")
        

image_path = "videos\\YOLOData\\test\\images\\video13_14010_jpg.rf.7ec9ee7f625a1fd651aad650ff5a6f4b.jpg"
label_path = "videos\\YOLOData\\test\\labels\\video13_14010_jpg.rf.7ec9ee7f625a1fd651aad650ff5a6f4b.txt"
test_im = cv2.imread(image_path)
with open(label_path) as f:
    contents = f.readlines()
lab = np.asarray([float(x) for x in contents[0].split(" ")])
print(lab)
print(type(test_im.tolist()))
print(test_im.shape)
x_draw = int(float(lab[1])*test_im.shape[1])
y_draw = int(float(lab[2])*test_im.shape[0])
print(x_draw,y_draw)
test_im = cv2.circle(test_im, (x_draw,y_draw), radius=5, color=(0, 0, 255), thickness=-1)
cv2.imshow("image", test_im)
cv2.setMouseCallback('image', click_event)
cv2.waitKey(0)
cv2.destroyAllWindows()

[          0      0.4625     0.47847   0.0046875   0.0083333]
<class 'list'>
(720, 1280, 3)
592 344


# Image detection for each side #

Train YOLO Model to detect tennis ball

In [2]:
# Train YOLO Model
from ultralytics import YOLO
# model = YOLO('yolo26n.pt') # Standard YOLO26n model
model = YOLO("runs\\detect\\train\\weights\\best.pt") # My pretrained YOLO for tennis ball detection
print(model)
results = model.train(
    data='videos\\YOLOData\\data.yaml', 
    epochs=100, 
    imgsz=640,       # YOLO usually trains on square 640x640
    batch=8,        # Adjust based on your GPU memory
    device=0,         # Use 0 for GPU, 'cpu' if no GPU
    workers = 1
)


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed 
train: Fast image access  (ping: 14.22.4 ms, read: 5.22.6 MB/s, size: 107.5 KB)
train: Scanning D:\jupyter_server\TennisVision\videos\YOLOData\train\labels.cache... 21942 images, 84 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21942/21942  0.0s
train: D:\jupyter_server\TennisVision\videos\YOLOData\train\images\2497e41cea655ddc_jpg.rf.032961a34bfe8c0ea9262bcc454327f8.jpg: 1 duplicate labels removed
train: D:\jupyter_server\TennisVision\videos\YOLOData\train\images\2497e41cea655ddc_jpg.rf.0b69f20673b5171ae9121840a31e7d94.jpg: 1 duplicate labels removed
train: D:\jupyter_server\TennisVision\videos\YOLOData\train\images\2497e41cea655ddc_jpg.rf.37265815dfcbe74b13d49bac3ba1c6b7.jpg: 1 duplicate labels removed
train: D:\jupyter_server\TennisVision\videos\YOLOData\train\images\2497e41cea655ddc_jpg.rf.54c81d3b1079667dc7baa5d1de11d923.jpg: 1 duplicate labels removed
train: D:\jupyter_server\TennisVision\videos\YOLOData\train

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 7.0it/s 9.6s0.1s
                   all       1066       1555       0.74       0.54      0.604      0.326

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      1.54G      1.529     0.8624   0.004038         23        640: 100% ━━━━━━━━━━━━ 2743/2743 3.0it/s 15:23<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 2/67 4.0it/s 0.3s<16.2s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 8.0it/s 8.4s0.1s
                   all       1066       1555      0.824      0.637      0.702      0.405

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      1.54G      1.138     0.5324   0.001967         11        640: 100% ━━━━━━━━━━━━ 2743/2743 3.1it/s 14:37<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 2/67 4.1it/s 0.3s<16.0s

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 7.9it/s 8.5s0.1s
                   all       1066       1555      0.824      0.633        0.7      0.405

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    100/100      1.54G      1.124     0.5306   0.001954          7        640: 100% ━━━━━━━━━━━━ 2743/2743 3.1it/s 14:38<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 2/67 4.0it/s 0.3s<16.3s

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 7.9it/s 8.4s0.1s
                   all       1066       1555      0.826      0.632        0.7      0.404

100 epochs completed in 24.848 hours.
Optimizer stripped from D:\jupyter_server\TennisVision\runs\detect\train2\weights\last.pt, 5.4MB
Optimizer stripped from D:\jupyter_server\TennisVision\runs\detect\train2\weights\best.pt, 5.4MB

Validating D:\jupyter_server\TennisVision\runs\detect\train2\weights\best.pt...
Ultralytics 8.4.27  Python-3.8.3 torch-1.11.0 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6144MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 7.4it/s 9.1s0.1s
                   all       1066       1555      0.822      0.635        0.7      0.405
Speed: 0.2ms preprocess, 4.8ms inference, 0.0ms loss, 0.7ms postprocess

Test trained model

In [3]:
from ultralytics import YOLO
model = YOLO("runs\\detect\\train2\\weights\\best.pt")
all_image_paths = os.listdir(f"videos\\YOLOData\\test\\images")
for p in tqdm(all_image_paths[len(all_image_paths):len(all_image_paths)-500:-50]):
    results = model(f"videos\\YOLOData\\test\\images\\{p}")  # Predict on an image
    print(results[0].boxes.xyxy.cpu().numpy())
    results[0].show()  # Display results

  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


image 1/1 d:\jupyter_server\TennisVision\videos\YOLOData\test\images\video9_6740_jpg.rf.21ae0efa116bf1f20aeb1f0940e563fe.jpg: 384x640 2 tennis-balls, 61.3ms
Speed: 2.1ms preprocess, 61.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[[     807.48      248.22       816.4      257.77]
 [     1224.9      255.68      1233.9      264.61]]


 10%|█         | 1/10 [00:08<01:19,  8.88s/it]


image 1/1 d:\jupyter_server\TennisVision\videos\YOLOData\test\images\video9_2050_jpg.rf.7d26992703f6902733b63e2a9ecd7711.jpg: 384x640 1 tennis-ball, 12.1ms
Speed: 1.4ms preprocess, 12.1ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)
[[      627.7      120.47      634.16      126.76]]


...

 80%|████████  | 8/10 [00:19<00:04,  2.16s/it]


image 1/1 d:\jupyter_server\TennisVision\videos\YOLOData\test\images\TennisBall46_jpg.rf.6189e85b3121da2a8c19fe32923c5cb6.jpg: 384x640 1 tennis-ball, 13.3ms
Speed: 2.1ms preprocess, 13.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
[[     423.99      387.85      874.02      569.99]]


 90%|█████████ | 9/10 [00:21<00:01,  1.97s/it]


image 1/1 d:\jupyter_server\TennisVision\videos\YOLOData\test\images\fb7626f66f9ea8a2_jpg.rf.a7fc162ba63051990de5ee2628de3dab.jpg: 384x640 1 tennis-ball, 16.9ms
Speed: 1.9ms preprocess, 16.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
[[     515.32      190.67      768.12       438.4]]


100%|██████████| 10/10 [00:22<00:00,  2.28s/it]


Webcam Test

In [3]:
from ultralytics import YOLO
np.int = int 
np.float = float
np.bool = bool
model = YOLO("runs\\detect\\train2\\weights\\best.pt")
cam = cv2.VideoCapture(0)
while True:
    ret, frame = cam.read()
    results = model.track(frame,tracker="bytetrack.yaml", persist=True , verbose = False , iou=0.5)
    if len(results[0].boxes) > 0:
        box = results[0].boxes.xyxy[0].cpu().numpy()
        print(box)
        xB = int(box[2])
        xA = int(box[0])
        yB = int(box[3])
        yA = int(box[1])
        centerx = xA + (xB - xA) // 2
        centery = yA + (yB - yA) // 2
        topx , topy  = centerx , yA
        bottomx , bottomy = centerx , yB
        leftx , lefty = xA , centery
        rightx , righty = xB , centery 
        frame = cv2.circle(frame, (centerx,centery), radius=5, color=(0, 0, 255), thickness=-1)
        frame = cv2.circle(frame, (topx , topy), radius=1, color=(0, 255, 255), thickness=-1)
        frame = cv2.circle(frame, (bottomx , bottomy), radius=1, color=(0, 255, 255), thickness=-1)
        frame = cv2.circle(frame, (leftx , lefty), radius=1, color=(0, 255, 255), thickness=-1)
        frame = cv2.circle(frame, (rightx , righty), radius=1, color=(0, 255, 255), thickness=-1)
    else:
        im = frame
    cv2.imshow('Camera', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()
    

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


[     137.58      215.12      296.64      396.99]
[     137.58      215.12      296.64      396.99]
[     158.83      182.55      307.23      345.54]
[     158.83      182.55      307.23      345.54]
[     172.19      175.68      311.64      328.71]
[     180.62      172.69       311.3      315.85]
[     182.47      172.05      311.18      312.77]
[     186.62      170.85      316.25      312.32]
[     189.45      171.58      318.91      312.59]
[     190.09      171.93      319.83      312.96]
[     191.15         174       320.2       313.9]
[     191.25      174.78      320.36      314.39]
[     191.44      177.65      319.97      315.85]
[     192.44      179.68      322.09      317.42]
[     192.24      180.41      323.28      318.05]
[     188.38      180.15      323.92       321.5]
[     186.62      180.03      324.41       322.8]
[     181.12      180.58      320.02      322.81]
[     178.55      180.77      318.78      322.82]
[     172.92      180.03       313.4      321.58]


Video Test

In [ ]:
from ultralytics import YOLO
model = YOLO("runs\\detect\\train2\\weights\\best.pt")
video_path = "videos\\You respecting it rage quitting 😅 #swingvision #tennis #tennisplayer #tennisapp - SwingVision (720p, h264).mp4"
# video_path = "videos\\Jannik Sinner makes it look effortless🪄 - SwingVision (144p, h264).mp4"
cam = cv2.VideoCapture(video_path)
while True:
    ret, frame = cam.read()
    if not ret:break
    results = model.track(frame,tracker="bytetrack.yaml", persist=True , verbose = False , iou=0.5)
    print(results)
    if len(results[0].boxes) > 0:
        box = results[0].boxes.xyxy[0].cpu().numpy()
        xB = int(box[2])
        xA = int(box[0])
        yB = int(box[3])
        yA = int(box[1])
        centerx = xA + (xB - xA) // 2
        centery = yA + (yB - yA) // 2
        topx , topy  = centerx , yA
        bottomx , bottomy = centerx , yB
        leftx , lefty = xA , centery
        rightx , righty = xB , centery 
        frame = cv2.circle(frame, (centerx,centery), radius=5, color=(0, 0, 255), thickness=-1)
        frame = cv2.circle(frame, (topx , topy), radius=1, color=(0, 255, 255), thickness=1)
        frame = cv2.circle(frame, (bottomx , bottomy), radius=1, color=(0, 255, 255), thickness=1)
        frame = cv2.circle(frame, (leftx , lefty), radius=1, color=(0, 255, 255), thickness=1)
        frame = cv2.circle(frame, (rightx , righty), radius=1, color=(0, 255, 255), thickness=1)
    else:
        im = frame
    cv2.imshow('Camera', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'tennis-ball'}
obb: None
orig_img: array([[[ 25,  64,  58],
        [ 25,  64,  58],
        [ 24,  63,  57],
        ...,
        [ 17,  68,  60],
        [ 26,  77,  69],
        [ 30,  81,  73]],

       [[ 22,  61,  55],
        [ 22,  61,  55],
        [ 21,  60,  54],
        ...,
        [ 16,  67,  59],
        [ 23,  74,  66],
        [ 26,  77,  69]],

       [[ 20,  59,  53],
        [ 20,  59,  53],
        [ 19,  58,  52],
        ...,
        [ 12,  63,  55],
        [ 16,  67,  59],
        [ 17,  68,  60]],

       ...,

       [[200, 127,  66],
        [200, 127,  66],
        [200, 127,  66],
        ...,
        [197, 124,  62],
        [197, 124,  62],
        [197, 124,  62]],

       [[200, 127,  66],
        [200, 127,  66],
        [200, 127,  66],
        ...,
        [197, 124,  62],
        [197, 124,  62],
       

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:973: error: (-215:Assertion failed) size.width>0 && size.height>0 in function 'cv::imshow'


: 

# Court Segmentation #

In [ ]:
# Train YOLO Model
from ultralytics import YOLO
# model = YOLO('yolo26n.pt') # Standard YOLO26n model
model = YOLO("yolo26n.pt") # My pretrained YOLO for tennis ball detection
print(model)
results = model.train(
    data='videos\\YOLOCourt\\data.yaml', 
    epochs=1000, 
    imgsz=640,       # YOLO usually trains on square 640x640
    batch=12,        # Adjust based on your GPU memory
    device=0,         # Use 0 for GPU, 'cpu' if no GPU
    workers = 1
)

Test Court Lines

In [11]:
from ultralytics import YOLO
np.int = int 
np.float = float
np.bool = bool
model = YOLO("runs\\detect\\train3\\weights\\best.pt")
video_path = "videos\\video_infer.gif"
cam = cv2.VideoCapture(video_path)
points_to_keep = [[False,False,True,True],[False , False , False , True],[False,False , False , False],[False , True , True , True],[False , False , False , False],[False , False , True , False],[False , False , True , False],[True , True , True , True]]
classes = ['bottom-dead-zone', 'court', 'left-doubles-alley', 'left-service-box', 'net', 'right-doubles-alley', 'right-service-box', 'top-dead-zone']
while True:
    ret, frame = cam.read()
    # Preprocessing
    if not ret :break
    results = model(frame,verbose = False,imgsz=1280, conf=0.15)
    for r in results:
        if len(r.boxes.xyxy) > 0:
                print("Line detected")
                pred_class = r.boxes.cls
                boxes = r.boxes.xyxy.cpu().numpy()
                for idx in range(8):
                    if idx in pred_class:
                        i = torch.where(pred_class == idx)[0]
                        if len(i) > 0: i= i[0]
                        i = i.item()
                        b = boxes[i]
                        xA = int(b[0])
                        yA = int(b[1])
                        xB = int(b[2])
                        yB = int(b[3])
                        # if points_to_keep[idx][0]:
                        frame = cv2.circle(frame, (xA, yA), radius=5, color=(i*10, 0, 255 - i* 5), thickness=-1)
                        # if points_to_keep[idx][1]:
                        frame = cv2.circle(frame, (xB , yA), radius=5, color=(i*10, 0, 255- i* 5), thickness=-1)
                        # if points_to_keep[idx][2]:
                        frame = cv2.circle(frame, (xB, yB), radius=5, color=(i*10, 0, 255- i* 5), thickness=-1)
                        # if points_to_keep[idx][3]:
                        frame = cv2.circle(frame, (xA , yB), radius=5, color=(i*10, 0, 255- i* 5), thickness=-1)
        else:
                im = frame
    cv2.imshow('Camera', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line detected
Line d

In [10]:
from ultralytics import YOLO
np.int = int 
np.float = float
np.bool = bool
import pandas as pd

model = YOLO("runs\\detect\\train3\\weights\\best.pt")
video_path = "videos\\KaggleDataset\\test\\images\\video10.mp4"
cam = cv2.VideoCapture(video_path)
points_to_keep = [[False,False,True,True],[False , False , False , True],[False,False , False , False],[False , True , True , True],[False , False , False , False],[False , False , True , False],[False , False , True , False],[True , True , True , True]]
classes = ['bottom-dead-zone', 'court', 'left-doubles-alley', 'left-service-box', 'net', 'right-doubles-alley', 'right-service-box', 'top-dead-zone']
points_list = pd.read_csv("videos\\KaggleDataset\\test\\labels\\video10_court.csv").iloc[0].to_list()[1:]
points = [[int(points_list[i] ), int(points_list[i+1])] for i in range(0,len(points_list),2)]
print(points)
while True:
    ret, frame = cam.read()
    if not ret :break
    for p in points:
        frame = cv2.circle(frame, (p[0] , p[1]), radius=5, color=(255, 0, 0), thickness=-1)
    cv2.imshow('Camera', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

[[1213, 520], [638, 528], [96, 757], [1746, 740]]


For the Kaggle dataset we want to preprocess the data to fit the YOLO ultalistic model so:
- Images instead of videos 
- Anotate every frame using the annotation on the dataset
I organized the Kaggle data inside kaggle_format

In [ ]:

for dataset in ["train","valid","test"]:
    image_path = f"videos\\KaggleDataset\\kaggle_format\\{dataset}\\images"
    labels_path = f"videos\\KaggleDataset\\kaggle_format\\{dataset}\\labels"
    video_list = os.listdir(image_path)
    for video_name in video_list:
        video_path = f"{image_path}\\{video_name}"
        old_label_path = f"{labels_path}\\{video_name[:-4]}_court.csv"
        old_label_file = pd.read_csv(old_label_path)
        video_label = old_label_file.iloc[0].to_list()[1:]
        points_list = pd.read_csv(old_label_path).iloc[0].to_list()[1:]
        points = [[int(points_list[i]), int(points_list[i+1])] for i in range(0,len(points_list),2)]
        new_label_path = f"videos\\KaggleDataset\\{dataset}\\labels"
        cam = cv2.VideoCapture(video_path)
        ret, frame = cam.read()
        if ret:
            img_w = frame.shape[1]
            img_h = frame.shape[0]
        cam.release()
        points_text = f"{0} {points[0][0] / img_w} {points[0][1] / img_h} {points[1][0] / img_w} {points[1][1] / img_h} {points[2][0] / img_w} {points[2][1] / img_h} {points[3][0] / img_w} {points[3][1] / img_h}"
        print(img_w,img_h,points_text)
        cam = cv2.VideoCapture(video_path)
        frame_count = 0
        while True:
            ret, frame = cam.read()
            if not ret :break
            cv2.imwrite(f"videos\\KaggleDataset\\{dataset}\\images\\{video_name[:-4]}_frame{frame_count}.jpg",frame)
            with open(f"{new_label_path}\\{video_name[:-4]}_frame{frame_count}.txt","w") as f:
                f.write(points_text)
            frame_count += 1
    
        cam.release()

1920 1080 0 0.03333333333333333 0.8074074074074075 0.940625 0.8009259259259259 0.6177083333333333 0.5564814814814815 0.36041666666666666 0.5537037037037037
1920 1080 0 0.9625 0.6861111111111111 0.6635416666666667 0.4898148148148148 0.35833333333333334 0.48148148148148145 0.033854166666666664 0.6888888888888889


Try to train on new preprocessed data for YOLO

In [80]:

from ultralytics import YOLO
model = YOLO("runs\\segment\\train2\\weights\\best.pt")
# model = YOLO("runs\\segment\\train4\\weights\\best.pt")
results = model.train(
    data='videos\\KagglePlusCourtSegDataset\\data.yaml', 
    epochs=10, 
    imgsz=640,       # YOLO usually trains on square 640x640
    batch=12,        # Adjust based on your GPU memory
    device=0,         # Use 0 for GPU, 'cpu' if no GPU
    workers = 1
)

New https://pypi.org/project/ultralytics/8.4.35 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.32  Python-3.8.3 torch-1.11.0 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6144MiB)
WARNING Upgrade to torch>=2.0.0 for deterministic training.
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=videos\KagglePlusCourtSegDataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mod

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed 
train: Fast image access  (ping: 0.00.0 ms, read: 2524.6409.4 MB/s, size: 442.5 KB)
train: Scanning D:\jupyter_server\TennisVision\videos\KagglePlusCourtSegDataset\train\labels.cache... 5707 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5707/5707  0.0s
val: Fast image access  (ping: 0.00.0 ms, read: 1766.2587.3 MB/s, size: 435.4 KB)
val: Scanning D:\jupyter_server\TennisVision\videos\KagglePlusCourtSegDataset\valid\labels.cache... 1217 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1217/1217  0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 134 weight(decay=0.0), 154 weight(decay=0.00046875), 154 bias(decay=0.0)
Plotting labels to D:\jupyter_server\TennisVision\runs\segment\train9\labels.jpg... 
Image sizes 640 train, 640 val
Using 1 da

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.1it/s 16.2s0.4s
                   all       1217       1217      0.998      0.992      0.995       0.89      0.977      0.971      0.959       0.94

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/10       2.8G     0.3816     0.2009     0.2266   0.008785     0.0884          7        640: 100% ━━━━━━━━━━━━ 476/476 1.7it/s 4:41<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 2% ──────────── 1/51 1.0s/it 0.3s<52.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.4it/s 15.1s0.4s
                   all       1217       1217      0.973      0.962      0.992      0.978      0.973      0.962      0.992      0.978

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       9/10       2.8G     0.1389    0.09312    0.06511   0.003169    0.03927          7        640: 100% ━━━━━━━━━━━━ 476/476 1.8it/s 4:28<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 2% ──────────── 1/51 1.0it/s 0.3s<49.3s

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.4it/s 15.1s0.4s
                   all       1217       1217          1      0.999      0.995      0.946          1      0.999      0.995       0.99

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      10/10       2.8G     0.1002    0.08873    0.05333   0.002211     0.0377          7        640: 100% ━━━━━━━━━━━━ 476/476 1.7it/s 4:38<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 2% ──────────── 1/51 1.0s/it 0.3s<51.7s

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.3it/s 15.2s0.4s
                   all       1217       1217      0.997      0.997      0.995      0.982      0.997      0.997      0.995       0.99

10 epochs completed in 0.807 hours.
Optimizer stripped from D:\jupyter_server\TennisVision\runs\segment\train9\weights\last.pt, 6.5MB
Optimizer stripped from D:\jupyter_server\TennisVision\runs\segment\train9\weights\best.pt, 6.5MB

Validating D:\jupyter_server\TennisVision\runs\segment\train9\weights\best.pt...
Ultralytics 8.4.32  Python-3.8.3 torch-1.11.0 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6144MiB)
YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.7it/s 13.8s0.3s
                   al

Test reshape points of kaggle format dataset to change the labels from doubles line crosses to singles lines and net lines crossings

In [81]:
from ultralytics import YOLO
standard_court = np.asarray([[0,0],[100,0],[0,100],[100,100]], dtype=np.float32)
model = YOLO("runs\\segment\\train6\\weights\\best.pt")
doubles_lines = False
video_path = "videos\\SwingVision Max 4K Example - D1 Level Match at the 2024 KPSF Open - Austin Conlon (720p, h264).mp4"
cam = cv2.VideoCapture(video_path)
classes = ['bottom-dead-zone', 'court', 'left-doubles-alley', 'left-service-box', 'net', 'right-doubles-alley', 'right-service-box', 'top-dead-zone']
while True:
    ret, frame = cam.read()
    # Preprocessing
    if not ret :break
    results = model(frame,verbose = False)
    for r in results:
        masks = r.masks # The segmentation masks
        if masks:
            segments = masks.xy
            for seg in segments:
                hull = cv2.convexHull(seg)
                epsilon = 0.05 * cv2.arcLength(hull, True)
                approx_corners = cv2.approxPolyDP(hull, epsilon, True)
                if len(approx_corners) == 4:
                    corners = approx_corners.reshape(-1, 2)
                    if doubles_lines:
                        H = cv2.getPerspectiveTransform(np.asarray(corners, dtype=np.float32),standard_court)
                        H_inv = np.linalg.inv(H)
                        presp_corners = np.round(cv2.perspectiveTransform(np.asarray(corners, dtype=np.float32).reshape(-1, 1, 2),H).reshape(-1, 2),2)
                        presp_corners[0][0] = presp_corners[0][0] + 12.5
                        presp_corners[1][0] = presp_corners[1][0] - 12.5
                        presp_corners[2][0] = presp_corners[2][0] + 12 * 625
                        presp_corners[3][0] = presp_corners[3][0] - 12 * 625
                        presp_corners[2][1] = presp_corners[2][1] + 10000
                        presp_corners[3][1] = presp_corners[3][1] + 10000
                        remade_corners = np.round(cv2.perspectiveTransform(np.asarray(presp_corners, dtype=np.float32).reshape(-1, 1, 2),H_inv).reshape(-1, 2),2)
                        corners = remade_corners
                    for p in corners:
                        frame = cv2.circle(frame, (int(p[0]) , int(p[1])), radius=5, color=(0, 0, 255), thickness=-1)
    cv2.imshow('Camera', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

c:\Users\polis\anaconda3\lib\site-packages\ultralytics\nn\modules\head.py:246: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  idx = ori_index[torch.arange(batch_size)[..., None], index // nc]  # original index


Combine court segmentation (doubles lines ) and Kaggle Dataset (singles lines ) to a singular (singels lines) labeled dataset

In [36]:
# Copy Kaggle Dataset
for dataset in ["train","valid","test"]:
    image_path = f"videos\\KaggleDataset\\{dataset}\\images"
    labels_path = f"videos\\KaggleDataset\\{dataset}\\labels"
    new_image_path = f"videos\\KagglePlusCourtSegDataset\\{dataset}\\images"
    new_labels_path = f"videos\\KagglePlusCourtSegDataset\\{dataset}\\labels"
    image_list = os.listdir(image_path)
    for image_name in image_list:
        image = cv2.imread(f"{image_path}\\{image_name}")
        with open(f"{labels_path}\\{image_name[:-4:]}.txt","r") as f:
            label = f.read()
        cv2.imwrite(f"{new_image_path}\\{image_name}",image)
        with open(f"{new_labels_path}\\{image_name[:-4:]}.txt","w") as f:
            f.write(label)
        

In [ ]:
# Copy Segmentation Dataset but change target coordinates to singles lines and net lines
kill = False
standard_court = np.asarray([[0,0],[100,0],[0,100],[100,100]], dtype=np.float32)
for dataset in ["train","valid","test"]:
    image_path = f"videos\\CourtSegmentationDataset\\{dataset}\\images"
    labels_path = f"videos\\CourtSegmentationDataset\\{dataset}\\labels"
    new_image_path = f"videos\\KagglePlusCourtSegDataset\\{dataset}\\images"
    new_labels_path = f"videos\\KagglePlusCourtSegDataset\\{dataset}\\labels"
    image_list = os.listdir(image_path)
    for image_name in tqdm(image_list):
        image = cv2.imread(f"{image_path}\\{image_name}")
        with open(f"{labels_path}\\{image_name[:-4:]}.txt","r") as f:
            label = f.read()
        # Changing labels to correct format 
        label = label.split(" ")[1:]
        mask = [[float(label[i]), float(label[i+1])] for i in range(0,len(label),2)]
        if len(label) < 4: continue
        hull = cv2.convexHull(np.asarray(mask, dtype=np.float32))
        epsilon = 0.05 * cv2.arcLength(hull, True)
        approx_corners = cv2.approxPolyDP(hull, epsilon, True)
        if len(approx_corners) == 4:
            label = approx_corners.reshape(-1, 2)
            # print("Original",label)
            ind = np.lexsort((label[:, 1], label[:, 0]))
            label = label[ind]
            label_ind = [3,0,1,2]
            label = label[label_ind]
            # print("Sorted",label)
            H = cv2.getPerspectiveTransform(np.asarray(label, dtype=np.float32),standard_court)
            H_inv = np.linalg.inv(H)
            presp_corners = np.round(cv2.perspectiveTransform(np.asarray(label, dtype=np.float32).reshape(-1, 1, 2),H).reshape(-1, 2),2)
            presp_corners[0][0] = 12.5
            presp_corners[1][0] = 87.5
            presp_corners[2][0] = 7500
            presp_corners[3][0] = -7500
            presp_corners[2][1] =  10000
            presp_corners[3][1] =  10000
            remade_corners = np.round(cv2.perspectiveTransform(np.asarray(presp_corners, dtype=np.float32).reshape(-1, 1, 2),H_inv).reshape(-1, 2),2)
            # for p in mask:
            #     image = cv2.circle(image, (int(p[0]*image.shape[1]) , int(p[1]*image.shape[0])), radius=5, color=(0, 0, 255), thickness=-1)
            # for p in label:
            #     image = cv2.circle(image, (int(p[0]*image.shape[1]) , int(p[1]*image.shape[0])), radius=5, color=(0, 255, 0), thickness=-1)
            # for p in remade_corners:
            #     image = cv2.circle(image, (int(p[0]*image.shape[1]) , int(p[1]*image.shape[0])), radius=5, color=(255, 0, 0), thickness=-1)
            label = label[::-1]
            label_str = f"0 {label[0][0]} {label[0][1]} {label[1][0]} {label[1][1]} {label[2][0]} {label[2][1]} {label[3][0]} {label[3][1]}"
            # print("To write:",label)
            cv2.imwrite(f"{new_image_path}\\{image_name}",image)
            with open(f"{new_labels_path}\\{image_name[:-4:]}.txt","w") as f:
                f.write(label_str)
        # Visualize for debug
        #     cv2.imshow("image",image)
        #     while not cv2.waitKey(1) == ord('q'):
        #         if cv2.waitKey(1) == ord('w'):
        #             kill = True
        #         else:
        #             pass
        #     cv2.destroyAllWindows()
        #     if kill:
        #         break
        # if kill:
        #     break
        

100%|██████████| 53/53 [00:01<00:00, 42.34it/s]
